# `%ingest` — queue folder ingests, commit as one batch

A worked example of the `%ingest` magic from `eea_datalakehouse.notebook.magics` — see
`docs/notebook-facade-for-data-scientists.md` for the design behind it.

**Before running this for real:** the kernel environment needs whatever
`eea_datalakehouse.dds_ingestion.credentials.load_creds`/`load_base_url` expect (same as
importing `FolderIngest` directly today). Install the extra this needs once:
`pip install "EEADataLakehouse[notebook]"`.

**This is a separate session from `%catalog`'s, on purpose.** A catalog operation can't run
before its target has actually been ingested, so the two were never one atomic batch — see
"Two sessions, not one" in the design doc. Ingest and commit here first; only afterwards, in
a fresh set of `%catalog` cells (see `catalog_session_example.ipynb`), tag or copy what just
landed.

In [ ]:
import eea_datalakehouse.notebook  # registers %catalog/%ingest — no %load_ext needed

`%load_ext eea_datalakehouse.notebook.magics` still works too, and is safe to run either
before or after the import above — whichever runs first registers the magics, the other is
a no-op (see `magics.load_ipython_extension`'s docstring).

## Quick reference

`%ingest help` (or `%ingest help()`) lists every `IngestSession` method with its signature
and a short description — handy when you don't remember an exact parameter name
mid-notebook.

In [ ]:
%ingest help()

## Queue one or more folder ingests

Each call only records the intent — `folder`/`target_catalog_path`/`data_format` plus
anything else `FolderIngest` accepts (`intent`, `table_name`, `sub_path`, ...). Nothing
uploads until `commit()`.

In [ ]:
%ingest ingest(folder="./bw_2026", target_catalog_path="bwd.reference", data_format="parquet", intent="read_only", table_name="water_temperature", sub_path="2026")

A second ingest into a different table, queued on the *same* session — both run when the
cell below commits:

In [ ]:
%ingest ingest(folder="./bw_stations_2026", target_catalog_path="bwd.reference", data_format="parquet", table_name="stations")

## Commit — runs every queued ingest in order

Unlike `%catalog`'s `commit()`, this one **cannot roll back** an ingest that already
succeeded — see the design doc and `docs/read-only-ingest-client-plan.md` for why (deleting
an ingested table's backing data needs a server-side operation this package does not have
yet). If the second ingest above were to fail, the first one's table stays exactly as
ingested — `retry=True` only covers resuming a single failed ingest
(`FolderIngest.retry`'s own resume-the-load-step / re-upload-under-a-new-session logic),
not undoing an earlier one.

In [ ]:
%ingest commit(retry=True)

## What a partial failure looks like

```
%ingest commit
# -> ingest error: ingest './bw_stations_2026' -> 'bwd.reference' failed (...); 1 earlier
#    ingest(s) in this batch already committed and CANNOT be undone
```

That message is deliberately explicit about what already landed — check
`IngestCommitError.succeeded` (or just the catalog) before deciding what to do about the one
that failed, rather than assuming the whole batch was undone.

## Now the catalog side, in a fresh session

Once the ingest above has actually committed, a *separate* `%catalog` batch can tag or copy
what just landed — see `catalog_session_example.ipynb`:

```
%catalog tag("bwd.reference.water_temperature", ["reviewed", "2026"])
%catalog commit
```